# A6b Keyword & TF-IDF Baselines — GOLD (CPU, human-verified labels)

Mengevaluasi baseline keyword + TF-IDF terhadap **label human-gold** (`gold.jsonl`)
memakai assignment split leakage-safe yang SAMA dengan silver (review -> split
tidak diubah; hanya label referensi yang diganti dari silver ke gold). Keyword
dihitung ulang terhadap gold; TF-IDF dilatih ulang pada gold train, threshold
di-tune pada gold validation, lalu gold test dievaluasi **sekali**.

Output terpisah dari artefak silver yang dibekukan: `keyword-gold-v1-test-metrics.json`
dan `tfidf-gold-v1-test-metrics.json` dengan `reference_label_type = "human_gold"`.

Ikuti `docs/reproducibility-runbook.md` dan `tools/adjudicator/README.md` sebelum eksekusi.
Prasyarat: `gold.jsonl` sudah di-freeze via `annotation-agreement` + `freeze-gold`.


## Step 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Konfigurasi path & parameter

In [2]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_GOLD_DIR = DRIVE_ROOT / "data" / "annotations" / "gold"
DRIVE_SPLIT_DIR = DRIVE_ROOT / "data" / "splits"

PROJECT_DIR = Path("/content/hackathon/ml")
GOLD_DIR = PROJECT_DIR / "data" / "annotations" / "gold"
SPLIT_DIR = PROJECT_DIR / "data" / "splits"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"

DRIVE_METRICS_DIR = DRIVE_ROOT / "metrics"
DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"

GOLD_FILE = "gold.jsonl"
SPLIT_FILES = [
    "train_silver_v1.jsonl",
    "validation_silver_v1.jsonl",
    "test_silver_v1.jsonl",
    "split_manifest_silver_v1.json",
]

print("Drive root:", DRIVE_ROOT)
print("Sumber gold :", DRIVE_GOLD_DIR / GOLD_FILE)
print("Sumber split:", DRIVE_SPLIT_DIR)
print("Artifact dir (lokal):", ARTIFACT_DIR)


Drive root: /content/drive/MyDrive/SIPATURE
Sumber gold : /content/drive/MyDrive/SIPATURE/data/annotations/gold/gold.jsonl
Sumber split: /content/drive/MyDrive/SIPATURE/data/splits
Artifact dir (lokal): /content/hackathon/ml/artifacts


## Step 3 — Clone repository dari GitHub

In [3]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


Return code: 0

Cloning into '/content/hackathon'...



## Step 4 — Verifikasi commit terbaru (git log)

In [4]:
%cd /content/hackathon/ml
!git log --oneline -3


/content/hackathon/ml
cf4ee2d (HEAD -> main, origin/main, origin/HEAD) feat: preliminary vs final score comparison (module + CLI + notebook 11)
e065b30 feat: gold baseline evaluation (package + CLI + notebook 10)
0d71fbc feat(adjudicator): highlight active annotator source when 'Pakai' is clicked


## Step 5 — Install dependencies

In [5]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


/content/hackathon/ml
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 610.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 6 — Verifikasi versi package

In [3]:
import numpy
import pandas
import pyarrow
import sklearn
import joblib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)


NumPy: 2.2.6
Pandas: 2.3.3
PyArrow: 19.0.1
Scikit-learn: 1.7.2
Joblib: 1.5.3


## Step 7 — Copy gold + split dari Drive ke lokal

In [4]:
# Salin gold.jsonl + split (train/validation/test + manifest) dari Drive.
import shutil
from pathlib import Path

GOLD_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

gold_source = DRIVE_GOLD_DIR / GOLD_FILE
assert gold_source.is_file(), (
    f"gold.jsonl tidak ditemukan di Drive: {gold_source}\n"
    "Upload gold.jsonl hasil freeze-gold ke SIPATURE/data/annotations/gold/ terlebih dahulu."
)
shutil.copy2(gold_source, GOLD_DIR / GOLD_FILE)
print("Disalin:", GOLD_FILE, "->", GOLD_DIR)

for filename in SPLIT_FILES:
    source = DRIVE_SPLIT_DIR / filename
    assert source.is_file(), f"Split file tidak ditemukan di Drive: {source}"
    shutil.copy2(source, SPLIT_DIR / filename)
    print("Disalin:", filename)


Disalin: gold.jsonl -> /content/hackathon/ml/data/annotations/gold
Disalin: train_silver_v1.jsonl
Disalin: validation_silver_v1.jsonl
Disalin: test_silver_v1.jsonl
Disalin: split_manifest_silver_v1.json


## Step 8 — Import modul sipature_ml

In [5]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


## Step 9 — Validasi gold & verifikasi split terkunci

In [7]:
import json
from pathlib import Path

from sipature_ml.config import load_config
from sipature_ml.manifest import sha256_file

taxonomy = load_config("taxonomy")

# 1) Validasi gold JSONL (gold tidak punya annotator_id -> cek label langsung).
records = [json.loads(line) for line in (GOLD_DIR / GOLD_FILE).read_text(encoding="utf-8").splitlines() if line]
invalid = []
for record in records:
    errors = []
    if record.get("annotation_status") not in {"completed", "adjudicated"}:
        errors.append(f"invalid status: {record.get('annotation_status')}")
    seen = set()
    for label in record.get("labels", []):
        aspect = label.get("aspect")
        if aspect not in taxonomy["aspect_definitions"]: errors.append(f"invalid aspect: {aspect}")
        if aspect in seen: errors.append(f"duplicate aspect: {aspect}")
        seen.add(aspect)
        polarity = label.get("polarity"); severity = label.get("severity")
        if polarity not in taxonomy["polarity_labels"]: errors.append(f"invalid polarity: {polarity}")
        if polarity == "negative" and severity not in taxonomy["severity_labels"]: errors.append(f"negative requires severity: {aspect}")
        if polarity != "negative" and severity is not None: errors.append(f"nonnegative severity must be null: {aspect}")
        if label.get("evidence_text") and label["evidence_text"] not in record.get("text", ""):
            errors.append(f"evidence not verbatim: {aspect}")
    if errors:
        invalid.append({"review_id": record.get("review_id"), "errors": errors})
print("Gold validation -> records:", len(records), "| invalid:", len(invalid))
assert not invalid, f"Gold invalid: {invalid}"

# 2) Verifikasi split manifest terkunci + hash split tidak berubah.
manifest = json.loads((SPLIT_DIR / "split_manifest_silver_v1.json").read_text(encoding="utf-8"))
assert manifest.get("test_is_locked"), "Split manifest tidak terkunci"
for split, output in manifest["outputs"].items():
    actual = sha256_file(SPLIT_DIR / output["path"])
    assert actual == output["sha256"], f"Hash split berubah: {split}"
print("Split locked & hash verifikasi OK.")

# 3) Verifikasi cakupan: review_id gold = review_id split (1320).
gold_ids = {r["review_id"] for r in records}
split_ids = set()
for split in ("train", "validation", "test"):
    split_ids |= {json.loads(line)["review_id"] for line in (SPLIT_DIR / f"{split}_silver_v1.jsonl").read_text(encoding="utf-8").splitlines() if line}
print("Gold records:", len(gold_ids), "| Split records:", len(split_ids), "| identical:", gold_ids == split_ids)
assert gold_ids == split_ids, "Cakupan review gold != split"

Gold validation -> records: 1320 | invalid: 0
Split locked & hash verifikasi OK.
Gold records: 1320 | Split records: 1320 | identical: True


## Step 10 — Jalankan evaluasi gold baselines

In [8]:
from sipature_ml.gold_baselines import run_gold_baselines

summary = run_gold_baselines(SPLIT_DIR, GOLD_DIR / GOLD_FILE, ARTIFACT_DIR)

print("Keyword Macro F1:", round(summary["keyword_test_macro_f1"], 4))
print("Keyword Micro F1:", round(summary["keyword_test_micro_f1"], 4))
print("TF-IDF  Macro F1:", round(summary["tfidf_test_macro_f1"], 4))
print("TF-IDF  Micro F1:", round(summary["tfidf_test_micro_f1"], 4))
print("TF-IDF representation:", summary["selected_tfidf_representation"])


Keyword Macro F1: 0.7797
Keyword Micro F1: 0.7102
TF-IDF  Macro F1: 0.6379
TF-IDF  Micro F1: 0.7445
TF-IDF representation: word_char


## Step 11 — Tampilkan perbandingan silver vs gold

In [9]:
import json
from pathlib import Path

gold_kw = json.loads((ARTIFACT_DIR / "metrics" / "keyword-gold-v1-test-metrics.json").read_text())
gold_tf = json.loads((ARTIFACT_DIR / "metrics" / "tfidf-gold-v1-test-metrics.json").read_text())

def silver_macro(key):
    path = ARTIFACT_DIR / "metrics" / f"{key}-silver-v1-test-metrics.json"
    if not path.is_file():
        return None
    return json.loads(path.read_text())["macro_f1"]

print(f"{'model':<10}{'silver':>10}{'gold':>10}{'delta':>10}")
for key, label, gold in (("keyword", "Keyword", gold_kw["macro_f1"]), ("tfidf", "TF-IDF", gold_tf["macro_f1"])):
    silver = silver_macro(key)
    if silver is not None:
        print(f"{label:<10}{silver:>10.4f}{gold:>10.4f}{gold-silver:>+10.4f}")

print("\nPer-aspect F1 (gold test):")
for aspect, info in summary["per_aspect"].items():
    print(f"  {aspect:<22} kw={info['keyword_f1']:.4f} tfidf={info['tfidf_f1']:.4f} (sup {info['support']})")


model         silver      gold     delta

Per-aspect F1 (gold test):
  access                 kw=0.4800 tfidf=0.5333 (sup 7)
  cleanliness            kw=0.9000 tfidf=0.8387 (sup 32)
  comfort                kw=0.3306 tfidf=0.7773 (sup 101)
  crowding               kw=0.8000 tfidf=0.3077 (sup 9)
  maintenance            kw=0.9655 tfidf=0.8000 (sup 14)
  opening_hours          kw=0.6667 tfidf=0.0000 (sup 1)
  parking                kw=0.9756 tfidf=0.8500 (sup 20)
  price_transparency     kw=0.7667 tfidf=0.5581 (sup 28)
  public_facilities      kw=0.9048 tfidf=0.7143 (sup 21)
  safety                 kw=0.9412 tfidf=0.5000 (sup 9)
  sanitation             kw=1.0000 tfidf=0.8108 (sup 19)
  scenery                kw=0.6667 tfidf=0.7714 (sup 88)
  staff_service          kw=0.5185 tfidf=0.5217 (sup 11)
  waste                  kw=1.0000 tfidf=0.9474 (sup 9)


## Step 12 — Copy output ke Drive

In [10]:
# Salin metrics + summary ke Drive (artefak persisten).
import shutil
from pathlib import Path

for local_dir, drive_dir in (
    (ARTIFACT_DIR / "metrics", DRIVE_METRICS_DIR),
    (ARTIFACT_DIR / "reports", DRIVE_REPORT_DIR),
):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for source in sorted(local_dir.glob("*gold-v1*")) + sorted(local_dir.glob("gold_baseline_summary.json")):
        if source.is_file():
            shutil.copy2(source, drive_dir / source.name)
            print(f"Disalin: {source.name} -> {drive_dir}")


Disalin: keyword-gold-v1-test-metrics.json -> /content/drive/MyDrive/SIPATURE/metrics
Disalin: tfidf-gold-v1-test-metrics.json -> /content/drive/MyDrive/SIPATURE/metrics
Disalin: gold_baseline_summary.json -> /content/drive/MyDrive/SIPATURE/reports


## Step 13 — Run summary (hash & metric)

In [11]:
# ============================================================
# RUN SUMMARY — hash, metric, dan limitations.
# ============================================================
import json
from pathlib import Path
from sipature_ml.manifest import sha256_file

print("REFERENCE LABEL TYPE:", summary["reference_label_type"])
print("SPLIT VERSION        :", summary["split_version"])
print("GOLD SHA256          :", summary["gold_sha256"])
print("Keyword Macro F1     :", round(summary["keyword_test_macro_f1"], 4))
print("TF-IDF  Macro F1     :", round(summary["tfidf_test_macro_f1"], 4))

print("\nOUTPUT METRICS DIR  :", ARTIFACT_DIR / "metrics")
print("OUTPUT REPORTS DIR   :", ARTIFACT_DIR / "reports")

print("\nREMINDER: metric ini adalah agreement terhadap HUMAN-GOLD labels.")
print("Keyword silver F1 0.9768 bersifat circular terhadap silver rules;")
print("gold F1 lebih rendah dan jujur. TF-IDF lebih robust (delta lebih kecil).")


REFERENCE LABEL TYPE: human_gold
SPLIT VERSION        : silver-split-1.0.0
GOLD SHA256          : 2376ede57eeec8b54b6548601621ca1157a1d94dfe4bf2b06007335db1e72aaa
Keyword Macro F1     : 0.7797
TF-IDF  Macro F1     : 0.6379

OUTPUT METRICS DIR  : /content/hackathon/ml/artifacts/metrics
OUTPUT REPORTS DIR   : /content/hackathon/ml/artifacts/reports

REMINDER: metric ini adalah agreement terhadap HUMAN-GOLD labels.
Keyword silver F1 0.9768 bersifat circular terhadap silver rules;
gold F1 lebih rendah dan jujur. TF-IDF lebih robust (delta lebih kecil).
